## Connect ML Model with API
#### Goal
Build an API that:
- Takes input (POST request)
- Sends it to ML model
- Returns prediction

Previously:

I built ML model → used in Jupyter Notebook

Now:

You will make your model usable by apps, websites, clients
### Flow of Today’s Project
```
Client (Postman)
        ↓
Send input data (JSON)
        ↓
Flask API
        ↓
ML Model Prediction
        ↓
Return result (JSON)
```


### Step 1: Simple ML Model (Example)

We’ll use a basic model:

👉 Predict if student Pass/Fail based on marks
#### 1 — Install library (first time only)

In terminal / Colab:

In [1]:
pip install scikit-learn

### 2 — Import libraries

In [2]:
from sklearn.linear_model import LogisticRegression
import numpy as np

### What is happening?

| Library              | Why we need it                          |
|----------------------|------------------------------------------|
| sklearn              | Ready-made ML models                     |
| LogisticRegression   | Model for Pass/Fail prediction           |
| numpy                | Work with numbers/arrays                 |

### 3 — Create Training Data

In [3]:
X = np.array([[30], [40], [50], [60], [70]])
y = np.array([0, 0, 0, 1, 1])

This is the MOST IMPORTANT PART.

#### X → Input (marks)
```
[[30],
 [40],
 [50],
 [60],
 [70]]
```
#### y → Output (labels)

| Value | Meaning     |
|-------|-------------|
| 0     | Fail        |
| 1     | Pass        |

So we are telling the computer:

 "These marks produced these results in the past"

This is called **training data**.

### 4 — Create the Model
model = LogisticRegression()

In [4]:
model = LogisticRegression()

This creates an empty brain

Right now it knows NOTHING.

### 5 — Train the Model

In [5]:
model.fit(X, y)

LogisticRegression()

This is where learning happens

Model now studies the pattern:
- Low marks → Fail
- High marks → Pass

It finds a decision boundary like:
```
Marks < 55  → Fail
Marks ≥ 55 → Pass
```
This process is called training.

In [6]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Training data
X = np.array([[30], [40], [50], [60], [70]])
y = np.array([0, 0, 0, 1, 1])  # 0 = Fail, 1 = Pass

model = LogisticRegression()
model.fit(X, y)

LogisticRegression()

In [10]:
# 1. Show the trained coefficients
print("Model coefficients:", model.coef_)
print("Model intercept:", model.intercept_)

Model coefficients: [[0.56531197]]
Model intercept: [-31.0921734]


In [11]:
# 2. Make a prediction
test_marks = np.array([[55]])   # example input
prediction = model.predict(test_marks)
print("Prediction for marks 55:", "Pass" if prediction[0] == 1 else "Fail")

Prediction for marks 55: Fail


In [12]:
# 3. Predict multiple values
test_data = np.array([[35], [45], [65], [75]])
predictions = model.predict(test_data)
print("Predictions:", predictions)

Predictions: [0 0 1 1]


### Step 2: Save Model (Optional but Important)

In [13]:
import pickle

pickle.dump(model, open('model.pkl', 'wb'))

###  Step 3: Create Flask API
```
from flask import Flask, request, jsonify
import pickle
import numpy as np

app = Flask(__name__)

# Load model
model = pickle.load(open('model.pkl', 'rb'))

@app.route('/')
def home():
    return "ML API is running!"

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    
    marks = data.get('marks')
    
    prediction = model.predict([[marks]])[0]
    
    result = "Pass" if prediction == 1 else "Fail"
    
    return jsonify({
        "marks": marks,
        "prediction": result
    })

if __name__ == '__main__':
    app.run(debug=True)
    ```


### Understand the Big Picture

Flow of this project:
```
Train Model → Save Model → Load Model in Flask → Predict via API
```
#### Why?

Because Flask runs separately from training code.

Real companies NEVER retrain model every time user sends request.

They:
- Train once
- Save model
- Reuse model
### 1 — Create model training file

Create a new file in VS Code:
```
train_model.py
```
Paste this 👇
```
# train_model.py
# STEP 1 — Import libraries
from sklearn.linear_model import LogisticRegression
import numpy as np
import pickle

# STEP 2 — Training Data (marks vs pass/fail)
X = np.array([[30], [40], [50], [60], [70]])
y = np.array([0, 0, 0, 1, 1])  # 0 = Fail, 1 = Pass

# STEP 3 — Create model
model = LogisticRegression()

# STEP 4 — Train model
model.fit(X, y)

# STEP 5 — Save model to file
pickle.dump(model, open("model.pkl", "wb"))

print("Model trained and saved successfully!")
```
### Run this file

Open terminal in VS Code:
```
python train_model.py
```
We will see:
```
Model trained and saved successfully!
```
Now check your folder → you will see:
```
model.pkl
```

### 2 — Create Flask API file

Create new file:
```
app.py
```
Now paste  API code.

Here is the correct final version:
```
from flask import Flask, request, jsonify
import pickle
import numpy as np

# Create Flask app
app = Flask(__name__)

# Load trained model (IMPORTANT)
model = pickle.load(open("model.pkl", "rb"))

# Home route
@app.route("/")
def home():
    return "ML API is running!"

# Prediction route
@app.route("/predict", methods=["POST"])
def predict():
    # Get JSON data sent by user
    data = request.get_json()

    # Extract marks value
    marks = data.get("marks")

    # Convert to numpy array for model
    marks_array = np.array([[marks]])

    # Make prediction
    prediction = model.predict(marks_array)[0]

    # Convert numeric result to text
    result = "Pass" if prediction == 1 else "Fail"

    # Send JSON response
    return jsonify({
        "marks": marks,
        "prediction": result
    })

# Run server
if __name__ == "__main__":
    app.run(debug=True)
```
### 3 — Run Flask API

In terminal:
```
python app.py
```
We will see:
```
Running on http://127.0.0.1:5000/
```
Open browser:
```
http://127.0.0.1:5000/
```
We will see:
```
ML API is running!
```
API is alive
### 4 — Test the /predict API

 Important: /predict is POST API

Browser only sends GET → so we must use Postman OR Python requests
### Test using Python (easy)

Create file:
```
test_api.py
```
Paste:
```
import requests

url = "http://127.0.0.1:5000/predict"

data = {"marks": 65}

response = requests.post(url, json=data)

print(response.json())
```
Run:
```
python test_api.py
```
### OUTPUT

We will see:
```
{'marks': 65, 'prediction': 'Pass'}
```


###  Step 4: Run API
```
python app.py
```

### Test the /predict API
### 1. Open your Flask project folder
Inside your project folder create a new folder named:
```
templates
```
Flask automatically looks for HTML inside this folder.

 project should look like:
```
 project/
│ app.py
│ model.pkl
└── templates/
```
### 2. Create HTML Form Page

Inside templates folder create file:
```
index.html
```
Paste this code 👇
```
<!DOCTYPE html>
<html>
<head>
    <title>Student Result Predictor</title>
</head>
<body>

    <h2>Enter Student Marks</h2>

    <form action="/predict" method="post">
        <input type="number" name="marks" placeholder="Enter marks" required>
        <button type="submit">Predict</button>
    </form>

    <h3>{{ prediction_text }}</h3>

</body>
</html>
```
This page sends POST request to /predict.
### 3.  Modify Flask Code (IMPORTANT)

Replace your current Flask code with this complete version:
```
from flask import Flask, request, jsonify, render_template
import pickle
import numpy as np

app = Flask(__name__)

# Load trained model
model = pickle.load(open('model.pkl', 'rb'))

# Home page → show form
@app.route('/')
def home():
    return render_template("index.html")

# Prediction route
@app.route('/predict', methods=['POST'])
def predict():
    marks = int(request.form['marks'])
    
    prediction = model.predict([[marks]])[0]
    result = "Pass" if prediction == 1 else "Fail"
    
    return render_template("index.html", prediction_text=f"Result: {result}")

if __name__ == '__main__':
    app.run(debug=True)
```
👉 Notice we replaced request.json with request.form.

Because browser form sends form data, not JSON.
###  4. Run Flask App

Open terminal in project folder and run:
```
python app.py
```
We will see:
```
Running on http://127.0.0.1:5000/
```
###  5. Open Browser

Open this URL:
```
http://127.0.0.1:5000/
```
### 6. Test It

1. Enter marks → example 65
2. Click Predict

We will see:
```
Result: Pass
```

### Step 5: Test Using Postman
Setup:
- Method: POST
- URL:
```
http://127.0.0.1:5000/predict
```
Body → JSON:
```
{
  "marks": 65
}
```
Response:
```
{
  "marks": 65,
  "prediction": "Pass"
}
```
### What Just Happened (Very Important)
1. You sent marks → API
2. API received using request.json
3. Data passed to ML model
4. Model predicted result
5. API returned JSON response